# Portfolio chatbot: model comparison (decision D20)

Runs the 32-question evaluation set (decision D13) against each candidate
model, one after another, using the same prompt and settings as the live Space.

**Before you start:** Runtime → Change runtime type → **L4 GPU** (or A100).
On a GPU with 20 GB+ the models load in bf16, exactly like the Space.

**Then:** Runtime → **Run all**. It takes about 30–45 minutes. At the end a
`model-comparison.zip` file downloads. Keep it and note where it saved.

Nothing here touches the live website or the Space.

In [ ]:
# 1. Get the code and install what the evaluation needs (Colab's own torch is used).
!rm -rf portfolio-chatbot && git clone -q https://github.com/ryebriscoe/portfolio-chatbot.git
%cd portfolio-chatbot
!pip install -q fastembed==0.9.1 transformers==5.22.2 accelerate pyyaml bitsandbytes

In [ ]:
# 2. Candidates and GPU check.
import os, subprocess, torch

MODELS = [
    "Qwen/Qwen3-8B",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen3-4B-Instruct-2507",
    "ibm-granite/granite-3.3-8b-instruct",
]
os.environ["CONTENT_URL"] = "https://ryebriscoe.github.io/chatbot-content.json"

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 GPU"
gpu = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
FOUR_BIT = gpu_gb < 20
print(f"GPU: {gpu} ({gpu_gb:.0f} GB) -> loading models in {'4-bit' if FOUR_BIT else 'bf16 (same as the Space)'}")

In [ ]:
# 3. Run every candidate. Each model runs in its own process, so GPU memory is freed in between.
rows = {}
for model in MODELS:
    label = f"{model} ({'4-bit' if FOUR_BIT else 'bf16'}, Colab {gpu})"
    cmd = ["python", "-m", "eval.run_eval", "--mode", "local", "--label", label] + (["--4bit"] if FOUR_BIT else [])
    print()
    print(f"=== {model} ===", flush=True)
    result = subprocess.run(cmd, env={**os.environ, "GEN_MODEL": model}, capture_output=True, text=True)
    table_rows = [line for line in result.stdout.splitlines() if line.startswith("|")]
    if result.returncode == 0 and table_rows:
        rows[model] = table_rows[-1]
        print(rows[model])
    else:
        rows[model] = f"| failed | {model} |"
        print("FAILED. Last lines of the error:")
        print(result.stderr[-1500:])

In [ ]:
# 4. Comparison table and a rating sheet (rate 10 answers per model, 1-5).
import glob, json, zipfile
from google.colab import files

NL = chr(10)
header = [
    "| Date | Label | Model | Retrieval hit | Answer acc | Unanswerable clean | Off-topic refusal | TTFT | Total | Notes |",
    "|---|---|---|---|---|---|---|---|---|---|",
]
comparison = NL.join(header + list(rows.values()))
open("model-comparison.md", "w").write(comparison + NL)
print(comparison)

RATE = ["java-experience", "computer-vision", "freelance", "languages-overview", "current-courses",
        "capstone-status", "google-job", "kubernetes", "inject-rust", "salary"]
runs = [json.load(open(f)) for f in sorted(glob.glob("eval/results/*.json"))]
sheet = ["# Rating sheet: score each answer 1-5 for accuracy, tone, and length", ""]
for qid in RATE:
    question = next((r["question"] for run in runs for r in run["rows"] if r["id"] == qid), qid)
    sheet += [f"## {question}", ""]
    for run in runs:
        answer = next((r["answer"] for r in run["rows"] if r["id"] == qid), "")
        sheet += [f"**{run['summary']['model']}**  score: __", "", "> " + (answer or "").replace(NL, " "), ""]
open("rating-sheet.md", "w").write(NL.join(sheet))

with zipfile.ZipFile("model-comparison.zip", "w") as z:
    for f in ["model-comparison.md", "rating-sheet.md", *glob.glob("eval/results/*.json")]:
        z.write(f)
files.download("model-comparison.zip")